In [1]:
import re
from pathlib import Path

def find_project_root(start):
    p = start.resolve()
    for _ in range(10):
        if (p / ".git").is_dir() or (p / "data" / "processed").is_dir():
            return p
        if p.parent == p:
            break
        p = p.parent
    raise RuntimeError("project root not found")

ROOT = find_project_root(Path.cwd())
print("ROOT:", ROOT)

ROOT: C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence


In [2]:
import re
from pathlib import Path

def find_project_root(start):
    p = start.resolve()
    for _ in range(10):
        if (p / ".git").is_dir() or (p / "data" / "processed").is_dir():
            return p
        if p.parent == p:
            break
        p = p.parent
    raise RuntimeError("project root not found")

ROOT = find_project_root(Path.cwd())
print("ROOT:", ROOT)
print()

# --- 1. Problem schema ---
print("=" * 70)
print("1. PROBLEM SCHEMA (src/tutor/schemas.py)")
print("=" * 70)
src = (ROOT / "src" / "tutor" / "schemas.py").read_text(encoding="utf-8")
m = re.search(r"(class Problem\b.*?)(?=\nclass |\Z)", src, re.DOTALL)
print(m.group(1) if m else "class Problem not found")

print()
print("=" * 70)
print("2. DRILL — where math_grader is called and where verdict is printed")
print("=" * 70)
drill = (ROOT / "src" / "tutor" / "drill.py").read_text(encoding="utf-8")
for i, line in enumerate(drill.splitlines(), start=1):
    if any(k in line for k in ("def run(", "def _save_attempt", "math_grader",
                                "Correct", "UNCERTAIN", "MATCH",
                                "anchor.question", "anchor.skill", "anchor.qid")):
        print(f"{i:4d}  {line}")

print()
print("=" * 70)
print("3. CLI — how the anchor is loaded and passed to drill")
print("=" * 70)
cli = (ROOT / "scripts" / "cli_tutor.py").read_text(encoding="utf-8")
for i, line in enumerate(cli.splitlines(), start=1):
    if any(k in line for k in ("_preloaded_problem", "drill", "run_drill",
                                "problem =", "anchor")):
        print(f"{i:4d}  {line}")

print()
print("=" * 70)
print("4. QUESTION_LOADER — how Problem objects are constructed")
print("=" * 70)
ql = (ROOT / "src" / "tutor" / "question_loader.py").read_text(encoding="utf-8")
for i, line in enumerate(ql.splitlines(), start=1):
    if any(k in line for k in ("def ", "Problem(", "question_id", "subquestion",
                                "document_id", "return")):
        print(f"{i:4d}  {line}")

ROOT: C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence

1. PROBLEM SCHEMA (src/tutor/schemas.py)
class Problem:
    problem_id: str
    skill_id: str
    topic: str
    subtopic: str
    structure_type: str
    prompt: str
    expected_answer: str
    topic_v2: Optional[str] = None


@dataclass

2. DRILL — where math_grader is called and where verdict is printed
  18  from .math_grader import grade, diagnose_wrong, MatchKind
 152  def _save_attempt(store, session_id, learner_id, skill_id,
 178      line = f"  Correct: {questions_correct}/{total_questions}"
 202  def run(engine, store, learner_id, session_id, anchor, logger=None,
 204      template_id = route_template(anchor.skill_id, anchor.prompt)
 206          print(f"  (No template for skill {anchor.skill_id} — skipping.)")
 214      topic_label = anchor.topic or anchor.skill_id.split(".")[0]
 218      print(f"  Drill: {anchor.skill_id}")
 247              "problem": _analogue_to_problem(analogue, template_id, anchor.sk

In [3]:
import re
from pathlib import Path
import sys

# Reuse ROOT from previous cell (if fresh notebook, recompute)
try:
    ROOT
except NameError:
    def find_project_root(start):
        p = start.resolve()
        for _ in range(10):
            if (p / ".git").is_dir() or (p / "data" / "processed").is_dir():
                return p
            if p.parent == p:
                break
            p = p.parent
        raise RuntimeError("project root not found")
    ROOT = find_project_root(Path.cwd())

sys.path.insert(0, str(ROOT))

print("=" * 70)
print("1. SAMPLE problem_id VALUES from loaded corpus")
print("=" * 70)
from src.tutor.question_loader import load_corpus, load_year_questions

# Full corpus
try:
    corpus = load_corpus()
    print(f"Corpus size: {len(corpus)}")
    print()
    print("First 10 problem_ids (any year):")
    for p in corpus[:10]:
        print(f"  {p.problem_id:<40s}  skill={p.skill_id}")
    print()
    # Filter to 2023
    y2023 = [p for p in corpus if "2023" in p.problem_id]
    print(f"2023-only problems: {len(y2023)}")
    for p in y2023[:8]:
        print(f"  {p.problem_id:<40s}  skill={p.skill_id}  prompt={p.prompt[:50]}")
except Exception as e:
    print("load_corpus failed:", e)

print()
print("=" * 70)
print("2. Year-specific loader — 2023 P1")
print("=" * 70)
try:
    probs = load_year_questions(2023, "P1")
    print(f"Count: {len(probs)}")
    for p in probs[:8]:
        print(f"  problem_id={p.problem_id!r}")
        print(f"      skill={p.skill_id}")
        print(f"      prompt={p.prompt[:70]}")
except Exception as e:
    print("load_year_questions failed:", e)

print()
print("=" * 70)
print("3. FULL MATCH branch in drill.py (lines 320-360)")
print("=" * 70)
drill_src = (ROOT / "src" / "tutor" / "drill.py").read_text(encoding="utf-8")
lines = drill_src.splitlines()
for i in range(319, min(360, len(lines))):
    print(f"{i+1:4d}  {lines[i]}")

print()
print("=" * 70)
print("4. _analogue_to_problem — how variant Problems get their id")
print("=" * 70)
m = re.search(r"(def _analogue_to_problem.*?)(?=\ndef |\Z)", drill_src, re.DOTALL)
if m:
    print(m.group(1))
else:
    print("not found")

1. SAMPLE problem_id VALUES from loaded corpus
Corpus size: 307

First 10 problem_ids (any year):
  2023_P1_Q1_1.1.1                          skill=algebra.quadratic.solve
  2023_P1_Q1_1.1.2                          skill=algebra.quadratic.solve
  2023_P1_Q1_1.1.3                          skill=algebra.surd.solve
  2023_P1_Q1_1.1.4                          skill=algebra.quadratic.inequality
  2023_P1_Q1_1.2                            skill=algebra.simultaneous.solve
  2023_P1_Q1_1.3                            skill=algebra.exponential.proof
  2023_P1_Q2_2.1.1                          skill=sequences.ap.term
  2023_P1_Q2_2.1.2                          skill=sequences.ap.sum
  2023_P1_Q2_2.1.3                          skill=sequences.ap.term
  2023_P1_Q2_2.2.1                          skill=sequences.quadratic.term

2023-only problems: 104
  2023_P1_Q1_1.1.1                          skill=algebra.quadratic.solve  prompt=Solve x^2+x-12=0
  2023_P1_Q1_1.1.2                          skill=a

In [4]:
from pathlib import Path

drill_path = ROOT / "src" / "tutor" / "drill.py"
src = drill_path.read_text(encoding="utf-8")

# --- Patch 1: add imports at the top of the file (after existing imports) ---
old_imports_anchor = "from .math_grader import grade, diagnose_wrong, MatchKind"
new_imports = (
    "from .math_grader import grade, diagnose_wrong, MatchKind\n"
    "from .step_grader import grade_steps as grade_steps_fn"
)
assert old_imports_anchor in src, "math_grader import anchor not found"
if "from .step_grader import" not in src:
    src = src.replace(old_imports_anchor, new_imports, 1)
    print("[ok] added step_grader import")
else:
    print("[skip] step_grader import already present")

# --- Patch 2: add helper function just before 'def run(' ---
helper = '''
# ============================================================
# Step-level feedback (Phase 5e)
# ============================================================

import re as _re


def _corpus_id_to_memo_id(problem_id: str) -> str:
    """2023_P1_Q1_1.1.1 -> 2023_P1_Q1.1.1 (drop underscore before sub-question)."""
    if not problem_id or problem_id.startswith("drill_"):
        return ""
    return _re.sub(r"Q(\\d+)_", r"Q\\1.", problem_id)


def _step_feedback(problem_id: str, learner_lines: list[str]) -> dict | None:
    """Return step-grader result for the anchor, or None if not applicable."""
    memo_id = _corpus_id_to_memo_id(problem_id)
    if not memo_id:
        return None
    try:
        r = grade_steps_fn(memo_id, learner_lines)
    except Exception:
        return None
    if r["total_marks"] == 0:
        return None
    return r


def _print_step_feedback(result: dict) -> None:
    """Format the step result for the CLI."""
    awarded = result["awarded_marks"]
    total = result["total_marks"]
    steps = result["steps"]

    missing = [s for s in steps if s.get("match_status") == "NOT_MATCHED"]
    uncertain = [s for s in steps if s.get("match_status") == "UNCERTAIN"]

    if not missing and not uncertain:
        print(f"  Full method shown — memo awards {total}/{total}.")
        return

    print(f"  Memo would award {awarded}/{total}.")
    if missing:
        parts = ", ".join(f"{s['desc']} ({s['marks']} mark{'s' if s['marks'] > 1 else ''})"
                          for s in missing)
        print(f"    Skipped: {parts}")
    if uncertain:
        parts = ", ".join(s["desc"] for s in uncertain)
        print(f"    Cannot verify: {parts}")
    print("    (In the exam, markers award marks per step.)")


'''
anchor = "def run(engine, store, learner_id, session_id, anchor"
assert anchor in src, "def run anchor not found"
if "_step_feedback" not in src:
    src = src.replace(anchor, helper + anchor, 1)
    print("[ok] added _step_feedback helper")
else:
    print("[skip] helper already present")

# --- Patch 3: insert call in MATCH branch, after the ✓ Correct print and before presentation notes ---
old_match = '''                print()
                print(f"  ✓ Correct.  ({questions_correct}/{i} this sitting)")

                # One NEW presentation note per question (dedup)'''

new_match = '''                print()
                print(f"  ✓ Correct.  ({questions_correct}/{i} this sitting)")

                # ---- Phase 5e: step-level feedback on the anchor ----
                step_result = _step_feedback(
                    drill_problem.problem_id,
                    [s.raw_text for s in submission.steps],
                )
                if step_result is not None:
                    _print_step_feedback(step_result)

                # One NEW presentation note per question (dedup)'''

assert old_match in src, "MATCH branch anchor not found"
src = src.replace(old_match, new_match, 1)
print("[ok] inserted step feedback call in MATCH branch")

drill_path.write_text(src, encoding="utf-8")
print()
print(f"drill.py: {len(src.splitlines())} lines")
print("OK")

[ok] added step_grader import
[ok] added _step_feedback helper
[ok] inserted step feedback call in MATCH branch

drill.py: 526 lines
OK


In [5]:
# Reload drill.py and check helpers exist
import importlib
import sys

# Remove cached modules so we pick up the new file
for m in list(sys.modules.keys()):
    if "drill" in m or "step_grader" in m:
        del sys.modules[m]

from src.tutor.drill import _corpus_id_to_memo_id, _step_feedback, _print_step_feedback

# Translation check
tests = [
    ("2023_P1_Q1_1.1.1", "2023_P1_Q1.1.1"),
    ("2023_P1_Q1_1.2",   "2023_P1_Q1.2"),
    ("2023_P1_Q2_2.1.1", "2023_P1_Q2.1.1"),
    ("2023_P2_Q3_3.1",   "2023_P2_Q3.1"),
    ("drill_quadratic_factored_abc123", ""),  # analogue -> empty
    ("", ""),
]
print("Translation test:")
ok = True
for src, expected in tests:
    got = _corpus_id_to_memo_id(src)
    mark = "✓" if got == expected else "✗"
    if got != expected:
        ok = False
    print(f"  {mark} {src!r:45s} -> {got!r:25s} (expected {expected!r})")

# Live step feedback check on Q1.1.1
print()
print("Live step feedback test on 2023_P1_Q1.1.1:")
result = _step_feedback("2023_P1_Q1_1.1.1", ["(x+4)(x-3)=0", "x=-4 or x=3"])
if result:
    print(f"  awarded: {result['awarded_marks']}/{result['total_marks']}")
    _print_step_feedback(result)
else:
    ok = False
    print("  [FAIL] no result returned")

print()
print("All translation checks passed." if ok else "SOME CHECKS FAILED.")

Translation test:
  ✗ '2023_P1_Q1_1.1.1'                            -> '2023_P1_Q1.1.1.1'        (expected '2023_P1_Q1.1.1')
  ✗ '2023_P1_Q1_1.2'                              -> '2023_P1_Q1.1.2'          (expected '2023_P1_Q1.2')
  ✗ '2023_P1_Q2_2.1.1'                            -> '2023_P1_Q2.2.1.1'        (expected '2023_P1_Q2.1.1')
  ✗ '2023_P2_Q3_3.1'                              -> '2023_P2_Q3.3.1'          (expected '2023_P2_Q3.1')
  ✓ 'drill_quadratic_factored_abc123'             -> ''                        (expected '')
  ✓ ''                                            -> ''                        (expected '')

Live step feedback test on 2023_P1_Q1.1.1:
  [FAIL] no result returned

SOME CHECKS FAILED.


In [6]:
drill_path = ROOT / "src" / "tutor" / "drill.py"
src = drill_path.read_text(encoding="utf-8")

old = '''def _corpus_id_to_memo_id(problem_id: str) -> str:
    """2023_P1_Q1_1.1.1 -> 2023_P1_Q1.1.1 (drop underscore before sub-question)."""
    if not problem_id or problem_id.startswith("drill_"):
        return ""
    return _re.sub(r"Q(\\d+)_", r"Q\\1.", problem_id)'''

new = '''def _corpus_id_to_memo_id(problem_id: str) -> str:
    """2023_P1_Q1_1.1.1 -> 2023_P1_Q1.1.1 (drop underscore + repeated digit)."""
    if not problem_id or problem_id.startswith("drill_"):
        return ""
    # Q<n>_<n>. -> Q<n>.
    return _re.sub(r"Q(\\d+)_\\1\\.", r"Q\\1.", problem_id)'''

assert old in src, "old regex block not found"
src = src.replace(old, new, 1)
drill_path.write_text(src, encoding="utf-8")
print("drill.py: regex patched")
print("OK")

drill.py: regex patched
OK


In [7]:
import subprocess, sys

test_code = """
from pathlib import Path
import sys as _sys
ROOT = Path.cwd()
_sys.path.insert(0, str(ROOT))

from src.tutor.drill import _corpus_id_to_memo_id, _step_feedback

# Translation
tests = [
    ("2023_P1_Q1_1.1.1", "2023_P1_Q1.1.1"),
    ("2023_P1_Q1_1.1.2", "2023_P1_Q1.1.2"),
    ("2023_P1_Q1_1.2",   "2023_P1_Q1.2"),
    ("2023_P1_Q1_1.3",   "2023_P1_Q1.3"),
    ("2023_P1_Q2_2.1.1", "2023_P1_Q2.1.1"),
    ("2023_P2_Q3_3.1",   "2023_P2_Q3.1"),
    ("2023_P2_Q5_5.1.1", "2023_P2_Q5.1.1"),
    ("drill_quadratic_factored_abc123", ""),
    ("", ""),
]
print('Translation:')
ok = True
for s, expected in tests:
    got = _corpus_id_to_memo_id(s)
    mark = 'Y' if got == expected else 'N'
    if got != expected:
        ok = False
    print(f'  [{mark}] {s!r:45s} -> {got!r}')

# Live step feedback
print()
print('Live step feedback:')
r = _step_feedback('2023_P1_Q1_1.1.1', ['(x+4)(x-3)=0', 'x=-4 or x=3'])
if r is None:
    print('  [FAIL] no result')
    ok = False
else:
    print(f'  awarded: {r[\"awarded_marks\"]}/{r[\"total_marks\"]}')
    for s in r['steps']:
        print(f'    step {s[\"index\"]}  {s[\"match_status\"]:12s}  {s[\"desc\"]}')

print()
print('ALL OK' if ok else 'SOME FAILED')
"""

result = subprocess.run(
    [sys.executable, "-c", test_code],
    capture_output=True, text=True, cwd=str(ROOT),
)
print(result.stdout)
if result.stderr:
    print("STDERR:", result.stderr)
print("Exit:", result.returncode)

Translation:
  [Y] '2023_P1_Q1_1.1.1'                            -> '2023_P1_Q1.1.1'
  [Y] '2023_P1_Q1_1.1.2'                            -> '2023_P1_Q1.1.2'
  [Y] '2023_P1_Q1_1.2'                              -> '2023_P1_Q1.2'
  [Y] '2023_P1_Q1_1.3'                              -> '2023_P1_Q1.3'
  [Y] '2023_P1_Q2_2.1.1'                            -> '2023_P1_Q2.1.1'
  [Y] '2023_P2_Q3_3.1'                              -> '2023_P2_Q3.1'
  [Y] '2023_P2_Q5_5.1.1'                            -> '2023_P2_Q5.1.1'
  [Y] 'drill_quadratic_factored_abc123'             -> ''
  [Y] ''                                            -> ''

Live step feedback:
  awarded: 2/3
    step 1  MATCHED       Factorise
    step 2  NOT_MATCHED   Set each factor to zero
    step 3  MATCHED       Write both roots

ALL OK

Exit: 0


In [9]:
import re
from pathlib import Path

def find_project_root(start):
    p = start.resolve()
    for _ in range(10):
        if (p / ".git").is_dir() or (p / "data" / "processed").is_dir():
            return p
        if p.parent == p:
            break
        p = p.parent
    raise RuntimeError("project root not found")

ROOT = find_project_root(Path.cwd())

print("=" * 70)
print("1. ANALOGUE_LIBRARY — what an analogue dict contains")
print("=" * 70)
lib = (ROOT / "src" / "tutor" / "analogue_library.py").read_text(encoding="utf-8")
# Show the loading + a sample record structure
for i, line in enumerate(lib.splitlines(), start=1):
    if any(k in line for k in ("def ", "problem", "expected", "roots", "factors",
                                "return", "load", "csv", "json")):
        print(f"{i:4d}  {line}")

print()
print("=" * 70)
print("2. ANALOGUE_GENERATORS — how a factorable-quadratic is generated")
print("=" * 70)
gen = (ROOT / "src" / "tutor" / "analogue_generators.py").read_text(encoding="utf-8")
# Find the factorable generator
m = re.search(r"(def .*quadratic_standard_factorable.*?)(?=\ndef |\Z)", gen, re.DOTALL)
if m:
    print(m.group(1)[:2500])
else:
    # Try to find any generator that mentions factored or standard_factorable
    for i, line in enumerate(gen.splitlines(), start=1):
        if "def " in line and ("factor" in line.lower() or "quadratic" in line.lower()):
            print(f"{i:4d}  {line}")

print()
print("=" * 70)
print("3. Sample analogue record (run the generator once)")
print("=" * 70)
import sys
sys.path.insert(0, str(ROOT))
try:
    from src.tutor.analogue_library import AnalogueLibrary
    # Try common signatures
    import inspect
    sig = inspect.signature(AnalogueLibrary.__init__)
    print(f"AnalogueLibrary.__init__ signature: {sig}")
    for name, fn in inspect.getmembers(AnalogueLibrary, predicate=inspect.isfunction):
        print(f"  method: {name}{inspect.signature(fn)}")
except Exception as e:
    print("AnalogueLibrary import failed:", e)

print()
print("=" * 70)
print("4. How drill.py fetches and uses analogues")
print("=" * 70)
drill = (ROOT / "src" / "tutor" / "drill.py").read_text(encoding="utf-8")
for i, line in enumerate(drill.splitlines(), start=1):
    if any(k in line for k in ("pool", "_analogue_to_problem", "analogue",
                                "template_id", "route_template")):
        print(f"{i:4d}  {line}")

print()
print("=" * 70)
print("5. Presentation matcher — trigger types")
print("=" * 70)
pres = (ROOT / "data" / "processed" / "tutor" / "presentation_rules_v1.csv").read_text(encoding="utf-8")
lines = pres.splitlines()
print(f"Header: {lines[0]}")
print()
print("Sample rows mentioning factorise / substitution / roots:")
for ln in lines[:200]:
    if any(k in ln.lower() for k in ("factor", "substitut", "root")):
        print(" ", ln[:180])
        if ln.count("factor") + ln.count("substitut") + ln.count("root") > 20:
            break

1. ANALOGUE_LIBRARY — what an analogue dict contains
   5    * data/processed/tutor/analogue_template_map.csv  (skill_id -> template_id)
  18  import csv
  24  MAP_PATH = ROOT / "data" / "processed" / "tutor" / "analogue_template_map.csv"
  30  def _load_template_map() -> dict:
  32          return {}
  35          reader = csv.DictReader(f)
  41      return mapping
  44  SKILL_TO_TEMPLATE = _load_template_map()
  53  def _stable_key(analogue: dict) -> str:
  54      """Hash the problem text so we can exclude already-shown analogues."""
  55      problem = str(analogue.get("problem", ""))
  56      return hashlib.md5(problem.encode("utf-8")).hexdigest()[:12]
  62  def get_template_for(skill_id: str):
  63      return SKILL_TO_TEMPLATE.get(str(skill_id).strip())
  66  def has_analogue(skill_id: str) -> bool:
  69          return False
  70      return len(POOL.get(tid, [])) > 0
  73  def all_skills() -> list:
  74      return sorted(SKILL_TO_TEMPLATE.keys())
  77  def pool_size(template

In [10]:
sg_path = ROOT / "src" / "tutor" / "step_grader.py"
src = sg_path.read_text(encoding="utf-8")

# Refactor: extract the inner-loop so both grade_steps and grade_steps_inline can use it.
# Add grade_steps_inline just before the final if __name__ block (or at end of file).

inline_fn = '''

def grade_steps_inline(steps: list[dict], learner_lines: list[str]) -> dict:
    """Same as grade_steps but takes pre-built steps (no CSV lookup).

    Used for drill variants that don't have a memo question_id.
    Steps must already have: index, desc, marks, forms, match_mode, is_final.
    """
    if not steps:
        return {
            "question_id": "<inline>",
            "total_marks": 0,
            "awarded_marks": 0,
            "steps": [],
            "feedback": "No step scheme.",
        }

    groups: dict[str, list] = {}
    for s in steps:
        groups.setdefault(s.get("method_group", "A"), []).append(s)

    best_awarded = -1
    best_results = None
    best_group = None
    for _name, g_steps in groups.items():
        awarded, results, _used = _grade_group(g_steps, learner_lines)
        if awarded > best_awarded:
            best_awarded = awarded
            best_results = results
            best_group = g_steps

    results = best_results
    awarded = best_awarded
    g_steps = best_group

    matched_values = {r["learner_line"] for r in results if r["matched"]}
    unmatched_lines = [ln for ln in learner_lines if ln not in matched_values]
    final_matched = any(r["matched"] and r["is_final"] for r in results)
    has_method_evidence = any(_METHOD_MARKERS.search(ln) for ln in unmatched_lines)

    for r in results:
        if r["matched"]:
            r["match_status"] = "MATCHED"
        elif final_matched and has_method_evidence:
            r["match_status"] = "UNCERTAIN"
        else:
            r["match_status"] = "NOT_MATCHED"

    total = sum(s["marks"] for s in g_steps)
    missing = [r for r in results if r["match_status"] == "NOT_MATCHED"]
    uncertain = [r for r in results if r["match_status"] == "UNCERTAIN"]

    if not missing and not uncertain:
        feedback = "Full marks: %d/%d" % (awarded, total)
    else:
        bits = ["missed '%s'" % m["desc"] for m in missing]
        if uncertain:
            bits.append("cannot verify " + ", ".join("'%s'" % u["desc"] for u in uncertain))
        feedback = "%d/%d. " % (awarded, total) + "; ".join(bits)

    return {
        "question_id": "<inline>",
        "total_marks": total,
        "awarded_marks": awarded,
        "steps": results,
        "feedback": feedback,
    }
'''

anchor = 'if __name__ == "__main__":'
if "def grade_steps_inline" not in src:
    src = src.replace(anchor, inline_fn.strip() + "\n\n\n" + anchor, 1)
    print("[ok] added grade_steps_inline")
else:
    print("[skip] grade_steps_inline already present")

sg_path.write_text(src, encoding="utf-8")
print(f"step_grader.py: {len(src.splitlines())} lines")
print("OK")

[ok] added grade_steps_inline
step_grader.py: 192 lines
OK


In [11]:
schemes_src = '''"""
Build step schemes from template + analogue, for drill variants.

Each builder returns a list of step dicts compatible with step_grader:
    index, desc, marks, forms, match_mode, is_final

Currently covers the quadratic family (factored / standard-factorable / formula).
Other templates return None and are skipped by the drill.
"""
from __future__ import annotations

import re
from typing import Optional


def _parse_roots(expected: str) -> list[str]:
    """Pull two numbers from an answer like 'x = -3 or x = 5' or 'x=-3 or x=5'."""
    if not expected:
        return []
    # Normalise
    s = expected.lower().replace(" ", "")
    # Common patterns
    nums = re.findall(r"x=(-?\\d+(?:\\.\\d+)?)", s)
    if len(nums) >= 2:
        return [nums[0], nums[1]]
    # Try "x = a; x = b"
    nums = re.findall(r"(-?\\d+(?:\\.\\d+)?)", s)
    if len(nums) >= 2:
        return [nums[0], nums[1]]
    return []


def _xterm_from_root(r: str) -> str:
    """For root r, produce 'x - r' or 'x + r'."""
    if r.startswith("-"):
        return f"x + {r[1:]}"
    return f"x - {r}"


def _build_factored(problem: str, expected: str) -> Optional[list[dict]]:
    """Factored-form quadratic: (x+a)(x+b)=0. Two marks: one per root."""
    roots = _parse_roots(expected)
    if len(roots) < 2:
        return None
    r1, r2 = roots[0], roots[1]
    return [
        {
            "index": 1, "desc": f"Root x = {r1}", "marks": 1,
            "forms": [f"x = {r1}", f"x={r1}", f"x = {r1} or x = {r2}"],
            "match_mode": "exact", "is_final": True, "method_group": "A",
        },
        {
            "index": 2, "desc": f"Root x = {r2}", "marks": 1,
            "forms": [f"x = {r2}", f"x={r2}", f"x = {r1} or x = {r2}",
                      f"x = {r2} or x = {r1}"],
            "match_mode": "exact", "is_final": True, "method_group": "A",
        },
    ]


def _build_standard_factorable(problem: str, expected: str) -> Optional[list[dict]]:
    """Standard-form factorable: x^2 + bx + c = 0. Three marks."""
    roots = _parse_roots(expected)
    if len(roots) < 2:
        return None
    r1, r2 = roots[0], roots[1]
    # Factorise step: acceptable forms
    f1 = f"(x - {r1})(x - {r2}) = 0"
    f2 = f"(x - {r2})(x - {r1}) = 0"
    # Set-to-zero step
    z1 = f"{_xterm_from_root(r1)} = 0"
    z2 = f"{_xterm_from_root(r2)} = 0"
    # Roots step
    roots_form = f"x = {r1} or x = {r2}"
    roots_form_rev = f"x = {r2} or x = {r1}"
    return [
        {
            "index": 1, "desc": "Factorise the quadratic", "marks": 1,
            "forms": [f1, f2,
                      f"(x-{r1})(x-{r2})=0", f"(x-{r2})(x-{r1})=0",
                      f"(x + {-int(r1)})(x + {-int(r2)}) = 0"
                      if r1.lstrip('-').isdigit() and r2.lstrip('-').isdigit() else f1],
            "match_mode": "exact", "is_final": False, "method_group": "A",
        },
        {
            "index": 2, "desc": "Set each factor to zero", "marks": 1,
            "forms": [f"{z1} or {z2}", f"{z2} or {z1}", z1, z2],
            "match_mode": "exact", "is_final": False, "method_group": "A",
        },
        {
            "index": 3, "desc": "Write both roots", "marks": 1,
            "forms": [roots_form, roots_form_rev,
                      roots_form.replace(" ", ""), roots_form_rev.replace(" ", "")],
            "match_mode": "exact", "is_final": True, "method_group": "A",
        },
    ]


_BUILDERS = {
    "quadratic_factored": _build_factored,
    "quadratic_standard_factorable": _build_standard_factorable,
    # add more templates here as we cover them
}


def build_scheme(template_id: str, analogue: dict, expected_answer: str) -> Optional[list[dict]]:
    """Return a step scheme for the given analogue, or None if unsupported."""
    builder = _BUILDERS.get(template_id)
    if builder is None:
        return None
    problem = analogue.get("problem", "")
    try:
        return builder(problem, expected_answer)
    except Exception:
        return None
'''

sgp = ROOT / "src" / "tutor" / "step_schemes.py"
sgp.write_text(schemes_src, encoding="utf-8")
print(f"step_schemes.py: {len(schemes_src.splitlines())} lines")
print("OK")

step_schemes.py: 115 lines
OK


In [12]:
sch_path = ROOT / "src" / "tutor" / "schemas.py"
src = sch_path.read_text(encoding="utf-8")

old = "    expected_answer: str\n    topic_v2: Optional[str] = None"
new = "    expected_answer: str\n    topic_v2: Optional[str] = None\n    step_scheme: Optional[list] = None"

if "step_scheme" not in src:
    assert old in src, "Problem expected_answer line not found"
    src = src.replace(old, new, 1)
    sch_path.write_text(src, encoding="utf-8")
    print("[ok] added step_scheme field to Problem")
else:
    print("[skip] step_scheme already present")

# Verify
m = __import__("re").search(r"class Problem\b.*?(?=\nclass |\Z)", src, __import__("re").DOTALL)
print()
print(m.group(0) if m else "(not found)")

[ok] added step_scheme field to Problem

class Problem:
    problem_id: str
    skill_id: str
    topic: str
    subtopic: str
    structure_type: str
    prompt: str
    expected_answer: str
    topic_v2: Optional[str] = None
    step_scheme: Optional[list] = None


@dataclass


In [13]:
drill_path = ROOT / "src" / "tutor" / "drill.py"
src = drill_path.read_text(encoding="utf-8")

# --- D1: import the new module ---
old_imp = "from .step_grader import grade_steps as grade_steps_fn"
new_imp = ("from .step_grader import grade_steps as grade_steps_fn, "
           "grade_steps_inline\nfrom .step_schemes import build_scheme")
if "grade_steps_inline" not in src:
    assert old_imp in src, "step_grader import anchor not found"
    src = src.replace(old_imp, new_imp, 1)
    print("[ok] imported grade_steps_inline + build_scheme")
else:
    print("[skip] inline grader already imported")

# --- D2: attach step_scheme when building a Problem from analogue ---
old_a2p = '''def _analogue_to_problem(analogue: dict, template_id: str, skill_id: str) -> Problem:
    return Problem(
        problem_id=f"drill_{template_id}_{uuid.uuid4().hex[:6]}",
        skill_id=skill_id,
        topic="",
        subtopic="",
        structure_type="routine_calculation",
        prompt=analogue["problem"],
        expected_answer=_extract_expected(analogue),
        topic_v2=None,
    )'''

new_a2p = '''def _analogue_to_problem(analogue: dict, template_id: str, skill_id: str) -> Problem:
    expected = _extract_expected(analogue)
    scheme = build_scheme(template_id, analogue, expected)
    return Problem(
        problem_id=f"drill_{template_id}_{uuid.uuid4().hex[:6]}",
        skill_id=skill_id,
        topic="",
        subtopic="",
        structure_type="routine_calculation",
        prompt=analogue["problem"],
        expected_answer=expected,
        topic_v2=None,
        step_scheme=scheme,
    )'''

if "step_scheme=scheme" not in src:
    assert old_a2p in src, "_analogue_to_problem block not found"
    src = src.replace(old_a2p, new_a2p, 1)
    print("[ok] _analogue_to_problem now attaches step_scheme")
else:
    print("[skip] already attached")

# --- D3: extend the MATCH branch to try inline scheme when CSV lookup fails ---
old_match = '''                # ---- Phase 5e: step-level feedback on the anchor ----
                step_result = _step_feedback(
                    drill_problem.problem_id,
                    [s.raw_text for s in submission.steps],
                )
                if step_result is not None:
                    _print_step_feedback(step_result)'''

new_match = '''                # ---- Phase 5e: step-level feedback ----
                # Try anchor (CSV lookup) first, fall back to inline scheme (variants)
                learner_lines_for_steps = [s.raw_text for s in submission.steps]
                step_result = _step_feedback(
                    drill_problem.problem_id,
                    learner_lines_for_steps,
                )
                if step_result is None and getattr(drill_problem, "step_scheme", None):
                    try:
                        step_result = grade_steps_inline(
                            drill_problem.step_scheme,
                            learner_lines_for_steps,
                        )
                    except Exception:
                        step_result = None
                if step_result is not None:
                    _print_step_feedback(step_result)'''

if "grade_steps_inline(" not in src:
    assert old_match in src, "MATCH branch step feedback block not found"
    src = src.replace(old_match, new_match, 1)
    print("[ok] MATCH branch now falls back to inline scheme")
else:
    print("[skip] MATCH branch already updated")

# --- D4: improve _print_step_feedback to show the memo line for missed steps ---
old_print = '''def _print_step_feedback(result: dict) -> None:
    """Format the step result for the CLI."""
    awarded = result["awarded_marks"]
    total = result["total_marks"]
    steps = result["steps"]

    missing = [s for s in steps if s.get("match_status") == "NOT_MATCHED"]
    uncertain = [s for s in steps if s.get("match_status") == "UNCERTAIN"]

    if not missing and not uncertain:
        print(f"  Full method shown — memo awards {total}/{total}.")
        return

    print(f"  Memo would award {awarded}/{total}.")
    if missing:
        parts = ", ".join(f"{s['desc']} ({s['marks']} mark{'s' if s['marks'] > 1 else ''})"
                          for s in missing)
        print(f"    Skipped: {parts}")
    if uncertain:
        parts = ", ".join(s["desc"] for s in uncertain)
        print(f"    Cannot verify: {parts}")
    print("    (In the exam, markers award marks per step.)")'''

new_print = '''def _print_step_feedback(result: dict) -> None:
    """Format the step result for the CLI, with help for missing steps."""
    awarded = result["awarded_marks"]
    total = result["total_marks"]
    steps = result["steps"]

    missing = [s for s in steps if s.get("match_status") == "NOT_MATCHED"]
    uncertain = [s for s in steps if s.get("match_status") == "UNCERTAIN"]

    if not missing and not uncertain:
        print(f"  Full method shown — memo awards {total}/{total}.")
        return

    print(f"  Memo would award {awarded}/{total}.")
    for s in missing:
        mark_word = "mark" if s["marks"] == 1 else "marks"
        print(f"    You skipped: {s['desc']}  ({s['marks']} {mark_word})")
        # Show what the memo wanted, from the FIRST acceptable form
        forms = s.get("forms") or []
        if forms:
            print(f"      The memo would show:   {forms[0]}")
    for u in uncertain:
        print(f"    Cannot verify: {u['desc']}")
    print("    (In the exam, markers award marks per step.)")'''

if "The memo would show" not in src:
    assert old_print in src, "_print_step_feedback block not found"
    src = src.replace(old_print, new_print, 1)
    print("[ok] _print_step_feedback now shows the memo's version of missed steps")
else:
    print("[skip] _print_step_feedback already updated")

drill_path.write_text(src, encoding="utf-8")
print(f"drill.py: {len(src.splitlines())} lines")
print("OK")

[ok] imported grade_steps_inline + build_scheme
[ok] _analogue_to_problem now attaches step_scheme
[ok] MATCH branch now falls back to inline scheme
[ok] _print_step_feedback now shows the memo's version of missed steps
drill.py: 543 lines
OK


In [14]:
import subprocess, sys

test_code = """
from pathlib import Path
import sys as _sys
ROOT = Path.cwd()
_sys.path.insert(0, str(ROOT))

from src.tutor.drill import _corpus_id_to_memo_id, _step_feedback, _analogue_to_problem, _print_step_feedback
from src.tutor.step_schemes import build_scheme
from src.tutor.step_grader import grade_steps_inline

print('=' * 60)
print('Test 1: anchor still works (CSV lookup path)')
print('=' * 60)
r = _step_feedback('2023_P1_Q1_1.1.1', ['x = -4 or x = 3'])
if r:
    print(f'  awarded: {r[\"awarded_marks\"]}/{r[\"total_marks\"]}')
    _print_step_feedback(r)
else:
    print('  [FAIL] anchor lookup returned None')

print()
print('=' * 60)
print('Test 2: factored-form variant, learner gave only answer')
print('=' * 60)
analogue = {
    'problem': 'Solve (x + 3)(x - 5) = 0',
    'steps': ['When a product is zero, at least one factor is zero.',
              'x + 3 = 0 gives x = -3',
              'x - 5 = 0 gives x = 5',
              'Answer: x = -3 or x = 5'],
}
scheme = build_scheme('quadratic_factored', analogue, 'x = -3 or x = 5')
print(f'  scheme built: {len(scheme)} steps')
r2 = grade_steps_inline(scheme, ['x = -3 or x = 5'])
print(f'  awarded: {r2[\"awarded_marks\"]}/{r2[\"total_marks\"]}')
_print_step_feedback(r2)

print()
print('=' * 60)
print('Test 3: standard-form factorable, learner skipped factorising')
print('=' * 60)
analogue2 = {
    'problem': 'Solve x^2 - 5x + 6 = 0',
    'steps': ['Factorise: (x - 2)(x - 3) = 0',
              'Set each factor to zero',
              'Answer: x = 2 or x = 3'],
}
scheme2 = build_scheme('quadratic_standard_factorable', analogue2, 'x = 2 or x = 3')
print(f'  scheme built: {len(scheme2)} steps')
r3 = grade_steps_inline(scheme2, ['x = 2 or x = 3'])
print(f'  awarded: {r3[\"awarded_marks\"]}/{r3[\"total_marks\"]}')
_print_step_feedback(r3)

print()
print('=' * 60)
print('Test 4: standard-form factorable, full working')
print('=' * 60)
r4 = grade_steps_inline(
    scheme2,
    ['(x - 2)(x - 3) = 0', 'x - 2 = 0 or x - 3 = 0', 'x = 2 or x = 3'],
)
print(f'  awarded: {r4[\"awarded_marks\"]}/{r4[\"total_marks\"]}')
_print_step_feedback(r4)
"""

result = subprocess.run(
    [sys.executable, "-c", test_code],
    capture_output=True, text=True, cwd=str(ROOT),
)
print(result.stdout)
if result.stderr:
    print("STDERR:", result.stderr)
print("Exit:", result.returncode)


STDERR: Traceback (most recent call last):
  File "<string>", line 7, in <module>
    from src.tutor.drill import _corpus_id_to_memo_id, _step_feedback, _analogue_to_problem, _print_step_feedback
  File "C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence\src\tutor\drill.py", line 19, in <module>
    from .step_grader import grade_steps as grade_steps_fn, grade_steps_inline
ImportError: cannot import name 'grade_steps_inline' from 'src.tutor.step_grader' (C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence\src\tutor\step_grader.py)

Exit: 1


In [15]:
sg_path = ROOT / "src" / "tutor" / "step_grader.py"
src = sg_path.read_text(encoding="utf-8")

print("Has grade_steps_inline:", "def grade_steps_inline" in src)
print("Has __main__ block:", '__name__ == "__main__"' in src)
print("File total lines:", len(src.splitlines()))
print()
print("Last 30 lines of step_grader.py:")
print("-" * 60)
for line in src.splitlines()[-30:]:
    print(line)
print("-" * 60)

Has grade_steps_inline: False
Has __main__ block: False
File total lines: 192

Last 30 lines of step_grader.py:
------------------------------------------------------------
    for r in results:
        if r["matched"]:
            r["match_status"] = "MATCHED"
        elif final_matched and has_method_evidence:
            r["match_status"] = "UNCERTAIN"
        else:
            r["match_status"] = "NOT_MATCHED"

    total = sum(s["marks"] for s in g_steps)
    missing = [r for r in results if r["match_status"] == "NOT_MATCHED"]
    uncertain = [r for r in results if r["match_status"] == "UNCERTAIN"]

    if not missing and not uncertain:
        feedback = "Full marks: %d/%d" % (awarded, total)
    elif uncertain and not missing:
        bits = ["cannot verify '%s' (alternate method used)" % u["desc"] for u in uncertain]
        feedback = "%d/%d. " % (awarded, total) + "; ".join(bits)
    else:
        bits = ["missed '%s' (%d mark)" % (m["desc"], m["marks"]) for m in missing]
    

In [16]:
sg_path = ROOT / "src" / "tutor" / "step_grader.py"
src = sg_path.read_text(encoding="utf-8")

if "def grade_steps_inline" in src:
    print("[skip] already present")
else:
    inline_fn = '''

def grade_steps_inline(steps: list[dict], learner_lines: list[str]) -> dict:
    """Same as grade_steps but takes pre-built steps (no CSV lookup).

    Used for drill variants that don't have a memo question_id.
    Steps must already have: index, desc, marks, forms, match_mode, is_final.
    """
    if not steps:
        return {
            "question_id": "<inline>",
            "total_marks": 0,
            "awarded_marks": 0,
            "steps": [],
            "feedback": "No step scheme.",
        }

    groups: dict[str, list] = {}
    for s in steps:
        groups.setdefault(s.get("method_group", "A"), []).append(s)

    best_awarded = -1
    best_results = None
    best_group = None
    for _name, g_steps in groups.items():
        awarded, results, _used = _grade_group(g_steps, learner_lines)
        if awarded > best_awarded:
            best_awarded = awarded
            best_results = results
            best_group = g_steps

    results = best_results
    awarded = best_awarded
    g_steps = best_group

    matched_values = {r["learner_line"] for r in results if r["matched"]}
    unmatched_lines = [ln for ln in learner_lines if ln not in matched_values]
    final_matched = any(r["matched"] and r["is_final"] for r in results)
    has_method_evidence = any(_METHOD_MARKERS.search(ln) for ln in unmatched_lines)

    for r in results:
        if r["matched"]:
            r["match_status"] = "MATCHED"
        elif final_matched and has_method_evidence:
            r["match_status"] = "UNCERTAIN"
        else:
            r["match_status"] = "NOT_MATCHED"

    total = sum(s["marks"] for s in g_steps)
    missing = [r for r in results if r["match_status"] == "NOT_MATCHED"]
    uncertain = [r for r in results if r["match_status"] == "UNCERTAIN"]

    if not missing and not uncertain:
        feedback = "Full marks: %d/%d" % (awarded, total)
    else:
        bits = ["missed '%s'" % m["desc"] for m in missing]
        if uncertain:
            bits.append("cannot verify " + ", ".join("'%s'" % u["desc"] for u in uncertain))
        feedback = "%d/%d. " % (awarded, total) + "; ".join(bits)

    return {
        "question_id": "<inline>",
        "total_marks": total,
        "awarded_marks": awarded,
        "steps": results,
        "feedback": feedback,
    }
'''
    src = src.rstrip() + "\n" + inline_fn
    sg_path.write_text(src, encoding="utf-8")
    print("[ok] appended grade_steps_inline")

# Verify
src2 = sg_path.read_text(encoding="utf-8")
print("Has grade_steps_inline:", "def grade_steps_inline" in src2)
print(f"File total lines: {len(src2.splitlines())}")

[ok] appended grade_steps_inline
Has grade_steps_inline: True
File total lines: 259


In [17]:
sg_path = ROOT / "src" / "tutor" / "step_grader.py"
src = sg_path.read_text(encoding="utf-8")

for name in ["def _grade_group", "_METHOD_MARKERS =", "def _line_matches",
             "def _norm", "def _tokens"]:
    print(f"  {'✓' if name in src else '✗'}  {name}")

  ✓  def _grade_group
  ✓  _METHOD_MARKERS =
  ✓  def _line_matches
  ✓  def _norm
  ✓  def _tokens


In [18]:
import subprocess, sys

test_code = """
from pathlib import Path
import sys as _sys
ROOT = Path.cwd()
_sys.path.insert(0, str(ROOT))

from src.tutor.drill import _corpus_id_to_memo_id, _step_feedback, _analogue_to_problem, _print_step_feedback
from src.tutor.step_schemes import build_scheme
from src.tutor.step_grader import grade_steps_inline

print('=' * 60)
print('Test 1: anchor still works (CSV lookup path)')
print('=' * 60)
r = _step_feedback('2023_P1_Q1_1.1.1', ['x = -4 or x = 3'])
if r:
    print(f'  awarded: {r[\"awarded_marks\"]}/{r[\"total_marks\"]}')
    _print_step_feedback(r)
else:
    print('  [FAIL] anchor lookup returned None')

print()
print('=' * 60)
print('Test 2: factored-form variant')
print('=' * 60)
analogue = {
    'problem': 'Solve (x + 3)(x - 5) = 0',
    'steps': [],
}
scheme = build_scheme('quadratic_factored', analogue, 'x = -3 or x = 5')
print(f'  scheme built: {len(scheme) if scheme else 0} steps')
if scheme:
    r2 = grade_steps_inline(scheme, ['x = -3 or x = 5'])
    print(f'  awarded: {r2[\"awarded_marks\"]}/{r2[\"total_marks\"]}')
    _print_step_feedback(r2)

print()
print('=' * 60)
print('Test 3: standard-form factorable, learner skipped factorising')
print('=' * 60)
analogue2 = {'problem': 'Solve x^2 - 5x + 6 = 0', 'steps': []}
scheme2 = build_scheme('quadratic_standard_factorable', analogue2, 'x = 2 or x = 3')
print(f'  scheme built: {len(scheme2) if scheme2 else 0} steps')
if scheme2:
    r3 = grade_steps_inline(scheme2, ['x = 2 or x = 3'])
    print(f'  awarded: {r3[\"awarded_marks\"]}/{r3[\"total_marks\"]}')
    _print_step_feedback(r3)

print()
print('=' * 60)
print('Test 4: standard-form factorable, full working')
print('=' * 60)
if scheme2:
    r4 = grade_steps_inline(
        scheme2,
        ['(x - 2)(x - 3) = 0', 'x - 2 = 0 or x - 3 = 0', 'x = 2 or x = 3'],
    )
    print(f'  awarded: {r4[\"awarded_marks\"]}/{r4[\"total_marks\"]}')
    _print_step_feedback(r4)
"""

result = subprocess.run(
    [sys.executable, "-c", test_code],
    capture_output=True, text=True, cwd=str(ROOT),
)
print(result.stdout)
if result.stderr:
    print("STDERR:", result.stderr)
print("Exit:", result.returncode)

Test 1: anchor still works (CSV lookup path)
  awarded: 1/3
  Memo would award 1/3.
    You skipped: Factorise  (1 mark)
    You skipped: Set each factor to zero  (1 mark)
    (In the exam, markers award marks per step.)

Test 2: factored-form variant
  scheme built: 2 steps
  awarded: 2/2
  Full method shown — memo awards 2/2.

Test 3: standard-form factorable, learner skipped factorising
  scheme built: 3 steps
  awarded: 1/3
  Memo would award 1/3.
    You skipped: Factorise the quadratic  (1 mark)
    You skipped: Set each factor to zero  (1 mark)
    (In the exam, markers award marks per step.)

Test 4: standard-form factorable, full working
  awarded: 3/3
  Full method shown — memo awards 3/3.

Exit: 0


In [19]:
spirit = '''# The Spirit of MethodMarker

*Source of truth. Read before every session.*

## The one-paragraph vision

I am trying to recreate myself — a teacher — but not bound to one classroom.
MethodMarker reaches many learners at once, at their exact point of need,
with the diagnosis a human tutor would give, the correction an examiner
expects, and the foundational reteaching a Grade 12 learner never got at
Grade 7. And because it works with learners every day, it learns them — by
name, by pattern, by history — and its understanding deepens over years
until it knows things no individual teacher could. Its discoveries are
reported to human teachers as referrals, so they too become better. Over
time, the tutor becomes a teaching archive: the accumulated expertise of
thousands of hours of tutoring, formalised, available to any classroom that
needs it.

## The spirit — my own words

"The tutor has to learn from the student errors also and to be able to
reteach those things and make corrections. A teacher learns from the
student and that becomes information to the teacher and a lesson to the
teacher. The tutor has to learn and create new solutions. The tutor is
human. The tutor should have a report of what it is learning and
discovering and what to teach. Its recommendations as time goes are far
richer than what the DBE is providing because it is working as a database
with students intently. That is why it is a tool that other teachers who
are human can use."

## What MethodMarker is

Not a tutor that explains mathematics to learners.
A tutor that **observes how this particular learner thinks**, identifies the
exact breakdown, reconstructs the missing foundation, tests whether the
repair worked, remembers the result, and changes how it teaches that learner.

The intelligence is not the ability to converse.
It is the ability to accumulate a memory of teaching.

## The four questions

Every feature, every session, every decision is judged by whether it
improves MethodMarker's ability to answer:

1. **What does this learner actually know?**
   Not what they claim. What the evidence demonstrates.

2. **Why did this learner fail this particular step?**
   Not "you made a mistake" but "this failure is consistent with X, and
   the evidence for that reading is Y."

3. **What should happen next?**
   Not "here is another explanation" but "given this learner's history,
   prerequisite structure, and previous intervention outcomes, this is the
   next appropriate teaching action."

4. **Did that teaching actually work?**
   Transfer. Not immediate performance. The killer question. If MethodMarker
   cannot answer this, it is not truly learning.

## The three learners

- **The student** learns mathematics.
- **The tutor** learns how to teach mathematics.
- **The teacher** learns from the tutor.

One flywheel. Each turn makes the others stronger.

## The four pillars

1. **Diagnosis per question.** Every question has a diagnosis report: what
   the memo wanted per step, what the learner showed, what mark was earned,
   what was lost, why. Backed by the DBE 12-year reports and memo mark
   schemes. Never a verdict without evidence.

2. **Fix the foundation, not the symptom.** When a learner fails a Grade 12
   step, the tutor walks the prerequisite chain to find the first broken
   link, and offers the drill that fixes the actual gap.

3. **Teach the way the examiner thinks.** Every presentation rule cites a
   specific DBE finding. Every piece of advice tied to a specific step.

4. **Set problems the way the examiner sets them.** Variants constrained
   by the fingerprint: number preferences, phrasing voice, mark shapes,
   answer forms. Practice that transfers because it is the exam's flavour.

## The Tutor Learning Engine

Not a discovery mechanism. A learning system.

    OBSERVE     the learner's actual working
       ->
    DIAGNOSE    which exact step broke, and why
       ->
    INTERVENE   the smallest action that might repair it
       ->
    MEASURE     did the repair transfer, days later
       ->
    LEARN       what worked, for whom, at which step
       ->
    UPDATE      how the tutor teaches next time

Discovery happens inside LEARN. It is one phase, not the system.

The LLM is one cognitive instrument inside this system. Not the system
itself. If the LLM is swapped tomorrow, the learning engine continues.

## Two outputs per session

**To the learner, now:** whatever diagnosis the tutor can produce at
whatever confidence. Help at the point of need. The learner does not wait
for validation.

**To the teacher, over time:** a referral, not analytics. "Across your 27
students, 14 are making the same sign error this week. Here is what I would
reteach."

## The student model

Every learner has a name and a profile. Their history. Their pattern. Their
personal model. When Tanaka opens the tutor tomorrow, it remembers Tanaka.

## The weekend test

Every proposed feature is judged by one question:

*Could another AI tutor add this in a weekend and become essentially the
same product?*

If yes, it is not our moat.

- LLM chat: no moat
- Voice tutor: no moat
- Generated worksheets: no moat
- Generic hints: no moat
- Fancy dashboard: no moat
- AI explanations: no moat

But:

- Three years of learner trajectories showing which misconceptions occur
  at which NSC steps, which prerequisite failures cause them, which
  interventions repair them, which interventions transfer, which persist,
  and which teachers confirmed those findings.

That is extremely difficult to reproduce. That is the archive. That is the
moat.

## The rules that protect the spirit

1. Never show a verdict without evidence.
2. The LLM is a thinking instrument, not the system. It can be swapped.
3. Presentation and diagnosis are different layers. Never conflated.
4. Every discovery is logged and permanent. Evidence preserved.
5. The teacher can always override.
6. The student is known by name, not by number.
7. Help is delivered at the moment of need.
8. Never optimise for being a better chatbot.

## What must never be lost

1. The tutor develops a **memory of teaching**, not just a knowledge of maths.
2. Two outputs per session: help the learner now, refer to the teacher over time.
3. The student is known by name. Per-learner, over time, at the point of need.
4. Interventions are measured for **transfer**, not just immediate improvement.
5. The teacher is a partner, not a customer.
6. Presentation and diagnosis are different layers.
7. I am recreating myself. One teacher, many classrooms. The tutor is me, multiplied.
8. The moat is the accumulated teaching archive. Not the engine.

## The sentence to keep visible during development

"The product gets better not merely when the code improves, but when the
tutor learns something from a learner that it can use again."

---

*Tanaka Peace Boanerge. ThunderMark. 2026.*
'''

(docs / "spirit.md").write_text(spirit, encoding="utf-8")
print(f"Wrote docs/spirit.md ({len(spirit.splitlines())} lines)")

NameError: name 'docs' is not defined